In [1]:
import torch
import yaml
from torch.utils.data import DataLoader

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded
from src.metrics import wasserstein
from src.model import HybridModel, _DiffusionBase

/home/justin/msc_ai/individual-project/diffusion-irregular-ehr/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

# Load data and confound it
train_ds, val_ds, test_ds, ytrain_std = load_ihdp(
    cfg.data.path,
    replication=1,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)
print("Confounder effect:", cfg.data.confounder_effect)
train_ds_conf, val_ds_conf, test_ds_conf = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect)
    for ds in (train_ds, val_ds, test_ds)
)

Confounder effect: 0.4


## Confouding and Standard Deviation

In [ ]:
print("Population Standard Deviations")
print("\t\tOriginal\tPost-flip\tChange")

for split, ds, ds_conf in zip(
    ("Train", "Val", "Test"),
    (train_ds, val_ds, test_ds),
    (train_ds_conf, val_ds_conf, test_ds_conf),
    strict=True,
):
    mu0_denormed = ds.mu0 * ytrain_std
    mu1_denormed = ds.mu1 * ytrain_std

    mu0_conf_denormed = ds_conf.mu0 * ytrain_std
    mu1_conf_denormed = ds_conf.mu1 * ytrain_std

    mu0_change = (mu0_conf_denormed.std() - mu0_denormed.std()) / mu0_denormed.std()
    mu1_change = (mu1_conf_denormed.std() - mu1_denormed.std()) / mu1_denormed.std()

    print(
        f"{split}\t$Y(0)$\t{mu0_denormed.std():.6f}"
        f"\t{mu0_conf_denormed.std():.6f}\t{mu0_change * 100:.1f}%"
    )
    print(
        f"\t$Y(1)$\t{mu1_denormed.std():.6f}"
        f"\t{mu1_conf_denormed.std():.6f}\t{mu1_change * 100:.1f}%"
    )

all_mu0_denormed = torch.cat([ds.mu0 for ds in (train_ds, val_ds, test_ds)]) * ytrain_std
all_mu1_denormed = torch.cat([ds.mu1 for ds in (train_ds, val_ds, test_ds)]) * ytrain_std

all_mu0_conf_denormed = (
    torch.cat([ds.mu0 for ds in (train_ds_conf, val_ds_conf, test_ds_conf)]) * ytrain_std
)
all_mu1_conf_denormed = (
    torch.cat([ds.mu1 for ds in (train_ds_conf, val_ds_conf, test_ds_conf)]) * ytrain_std
)

all_mu0_change = (
    all_mu0_conf_denormed.std() - all_mu0_denormed.std()
) / all_mu0_denormed.std()
all_mu1_change = (
    all_mu1_conf_denormed.std() - all_mu1_denormed.std()
) / all_mu1_denormed.std()

print(
    f"All\t$Y(0)$\t{all_mu0_denormed.std():.6f}"
    f"\t{all_mu0_conf_denormed.std():.6f}\t{all_mu0_change * 100:.1f}%"
)
print(
    f"\t$Y(1)$\t{all_mu1_denormed.std():.6f}"
    f"\t{all_mu1_conf_denormed.std():.6f}\t{all_mu1_change * 100:.1f}%"
)

Population Standard Deviations
		Original	Post-flip	Change
Train	$Y(0)$	1.33048	1.62853	22.4%
	$Y(1)$	0.46616	0.46327	-0.6%
Val	$Y(0)$	1.09330	1.38221	26.4%
	$Y(1)$	0.44873	0.45867	2.2%
Test	$Y(0)$	1.13358	1.53193	35.1%
	$Y(1)$	0.42721	0.44576	4.3%
All	$Y(0)$	1.26941	1.58020	24.5%
	$Y(1)$	0.45776	0.46018	0.5%


## Arm-split Evaluation

In [ ]:
y_both_conf = _DiffusionBase._assemble_yboth(
    train_ds_conf.a, train_ds_conf.y, train_ds_conf.y_cf
)

CLIP_VAL_CONF = y_both_conf.abs().max().item() * 2
print("CLIP_VAL_CONF:", CLIP_VAL_CONF)

CLIP_VAL_CONF: 7.191953659057617


In [ ]:
def evaluate_by_arm(
    model: HybridModel, loader: DataLoader, K: int, device="cpu", clip_val: float | None = None
):
    model.eval()
    all_y0, all_y1, all_a = [], [], []
    all_mu0, all_mu1 = [], []

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            a = batch["a"].to(device)
            y0_s, y1_s = model.sample_outcomes(x, a, K=K, clip_val=clip_val)
            all_y0.append(y0_s.cpu())
            all_y1.append(y1_s.cpu())
            all_a.append(a.cpu())
            all_mu0.append(batch["mu0"])
            all_mu1.append(batch["mu1"])

    y0 = torch.cat(all_y0)
    y1 = torch.cat(all_y1)
    a = torch.cat(all_a)
    mu0 = torch.cat(all_mu0)
    mu1 = torch.cat(all_mu1)

    out = {}
    for arm_name, mask in [("a=0", a == 0), ("a=1", a == 1)]:
        y0_0p5 = torch.quantile(y0[mask], 0.005, dim=1)
        y0_2p5 = torch.quantile(y0[mask], 0.025, dim=1)
        y0_97p5 = torch.quantile(y0[mask], 0.975, dim=1)
        y0_99p5 = torch.quantile(y0[mask], 0.995, dim=1)

        y1_0p5 = torch.quantile(y1[mask], 0.005, dim=1)
        y1_2p5 = torch.quantile(y1[mask], 0.025, dim=1)
        y1_97p5 = torch.quantile(y1[mask], 0.975, dim=1)
        y1_99p5 = torch.quantile(y1[mask], 0.995, dim=1)

        wd0, wd1 = wasserstein(
            y0[mask], y1[mask], mu0[mask], mu1[mask], sigma=1.0 / val_ds_conf.y_std
        )

        out[f"width_95_y0|{arm_name}"] = (y0_97p5 - y0_2p5).median().item()
        out[f"width_95_y1|{arm_name}"] = (y1_97p5 - y1_2p5).median().item()
        out[f"width_99_y0|{arm_name}"] = (y0_99p5 - y0_0p5).median().item()
        out[f"width_99_y1|{arm_name}"] = (y1_99p5 - y1_0p5).median().item()
        out[f"mean_abs(y0)|{arm_name}"] = y0[mask].abs().mean().item()
        out[f"mean_abs(y1)|{arm_name}"] = y1[mask].abs().mean().item()
        out[f"wasserstein(y0)|{arm_name}"] = wd0
        out[f"wasserstein(y1)|{arm_name}"] = wd1
    return out

In [ ]:
torch.manual_seed(cfg.train.seed)

model = HybridModel(cfg.model, cfg.diffusion)
model.load_state_dict(
    torch.load(
        "checkpoints/final_model_hybrid_conf_2026-08-21T12_46_12_rep1.pth", map_location="cpu"
    )
)
model.eval()

result = evaluate_by_arm(
    model,
    DataLoader(val_ds_conf, batch_size=cfg.train.batch_size),
    K=cfg.train.K,
    device="cpu",
    clip_val=CLIP_VAL_CONF,
)

for k in result:
    result[k] *= ytrain_std

result

{'width_95_y0|a=0': 35.056113817509186,
 'width_95_y1|a=0': 8.180246756439772,
 'width_99_y0|a=0': 35.08208283082922,
 'width_99_y1|a=0': 9.477273116394372,
 'mean_abs(y0)|a=0': 12.643843361304107,
 'mean_abs(y1)|a=0': 3.2368532636476743,
 'wasserstein(y0)|a=0': 11.8820812203835,
 'wasserstein(y1)|a=0': 1.2100720491058137,
 'width_95_y0|a=1': 35.054424889472784,
 'width_95_y1|a=1': 6.350793393637957,
 'width_99_y0|a=1': 35.0806800434932,
 'width_99_y1|a=1': 7.302046633346777,
 'mean_abs(y0)|a=1': 12.620738081335048,
 'mean_abs(y1)|a=1': 3.1951994362044616,
 'wasserstein(y0)|a=1': 11.923321241131394,
 'wasserstein(y1)|a=1': 0.8054201009923029}